# Project 5: Finetuning and Prompting

In this project, you will first learn how to use Huggingface's Transformers library to load large language models. Next, we will generate text from these models. Finally, we will work with a small text-to-SQL dataset, where the input is a natural language query and the output is a SQL query that can be executed against a database. You will explore (1) finetuning a pretrained language model, and (2) prompting the pretrained language model with examples. 

This project will be more open ended than the previous projects. We expect you to learn how to use the huggingface and torch documentation.

## Setup

First we install and import the required dependencies. These include:
* `torch` for modeling and training
* `transformers` for pre-trained models
* `datasets` from huggingface to load existing datasets.

In [1]:
%%capture
%pip install transformers datasets sentence-transformers accelerate

# Standard library imports
import torch
from torch.utils.data import Dataset, random_split
from transformers import AutoTokenizer, TrainingArguments, Trainer, AutoModelForCausalLM

Choose your device below. `auto` selects CUDA, then Apple MPS, then CPU. Use `.to(device)` for models and tensors throughout the notebook. CPU is available for debugging; full training depends on your machine's memory.

In Colab, choose **File → Save a copy in Drive** to save notebook edits. Allow the Drive mount below to save checkpoints and predictions in `MyDrive/CS183/HW5-2026/`.

In [2]:
DEVICE = "auto"  # "cuda": Colab/Kaggle/local NVIDIA; "mps": Mac GPU; "cpu": CPU

if DEVICE == "auto":
    DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
if DEVICE not in {"cuda", "mps", "cpu"}:
    raise ValueError("Choose auto, cuda, mps, or cpu.")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable: enable a GPU runtime or install CUDA-enabled PyTorch.")
if DEVICE == "mps" and not torch.backends.mps.is_available():
    raise RuntimeError("MPS unavailable on this machine; select cpu instead.")

device = torch.device(DEVICE)
TRAINER_DEVICE_ARGS = {"fp16": device.type == "cuda", "use_cpu": device.type == "cpu"}
print("Using device:", device)


Using device: cuda


In [3]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
OUTPUT_DIR = Path.cwd()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/CS183/HW5-2026")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving HW5 files to {OUTPUT_DIR}")

Saving HW5 files to d:\Aa GXT Files\Code\EECS 283A\hw5\eecs183-283A-26fall-hw5


## Loading Model

We will use GPT-2 Medium for this project. This includes both the GPT-2 tokenizer and the GPT-2 model weights itself. If you want to learn more about this model, you can read the GPT-2 paper https://d4mucfpksywv.cloudfront.net/better-language-models/language_models_are_unsupervised_multitask_learners.pdf.

Let's first load the tokenizer for the GPT-2 medium model. You can find how to do this by reading the documentation for AutoTokenzier in transformers, and finding the GPT-2 model of ~345 million params in there.

In [4]:
from transformers import AutoTokenizer
# Your code here
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2-medium")

tokenizer.pad_token = tokenizer.eos_token # convenient for padding later

d:\Aa GXT Files\Code\EECS 283A\hw5\eecs183-283A-26fall-hw5\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\GXT\.cache\huggingface\hub\models--openai-community--gpt2-medium. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Let's tokenize and detokenize some text from this model.

In [5]:
print(tokenizer.encode('Hello world'))
print(tokenizer.decode(tokenizer.encode('Hello world')))
print(tokenizer.encode("Hola, cómo estás😍"))

[15496, 995]
Hello world
[39, 5708, 11, 269, 10205, 5908, 1556, 40138, 47249, 235]


Now let's load the GPT-2 Medium model. Move the model to the selected device with `.to(device)`.

In [7]:
from transformers import AutoModelForCausalLM
# Your code here
gpt2_model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2-medium").to(device)

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 702.42it/s]


## Generate From the Model

Now let's generate some text from the model to test its LM capabilities. Let's first generate one output of length 50 tokens using greedy decoding (temperature = 0), which should get us some text with high likelihood under the model. When generating text, you can condition on phrases such as "The coolest thing in NLP right now is". Find the relevant function and arguments to use for generating text using the Huggingface documentation.

Hint: you may find https://huggingface.co/docs/transformers/main_classes/text_generation to be useful for learning about generating from LMs.

In [12]:
inputs = tokenizer("The coolest thing right now in NLP is", return_tensors="pt").input_ids.to(device)
# Your code here
sample_output = gpt2_model.generate(inputs, max_new_tokens=50, do_sample=False,pad_token_id=tokenizer.eos_token_id,)[0]

print("{}".format(tokenizer.decode(sample_output, skip_special_tokens=True)))

The coolest thing right now in NLP is the ability to use the same data to predict the future.

The future is always changing, and the data is always changing.

The future is always changing, and the data is always changing.

The future is always changing,


Now let's generate 10 pieces of random text of length 50 tokens from the model using random sampling with temperature set to 0.7. This will allow the text to be somewhat higher in diversity (random sampling) while maintaining reasonable quality (temperature < 1). 

In [14]:
inputs = tokenizer("The coolest thing right now in NLP is", return_tensors="pt").input_ids.to(device)
# Your code here
sample_outputs = gpt2_model.generate(inputs, max_new_tokens=50, num_return_sequences=10, do_sample=True,pad_token_id=tokenizer.eos_token_id, temperature=0.7)
# for i in range(10):
#     sample_outputs.append(gpt2_model.generate(inputs, max_new_tokens=50, return_dict_in_generate=True, output_scores=True, num_return_sequences=10, temperature=0))

for i, sample_output in enumerate(sample_outputs):
    print("{}: {}".format(i, tokenizer.decode(sample_output, skip_special_tokens=True)))

0: The coolest thing right now in NLP is that there are many different types of data.

Most of the data is available in machine learning, but I'm not sure if there is enough of it to make this post worth a post. You can find some examples of different types of data
1: The coolest thing right now in NLP is the ability to build more powerful models that can handle multiple datasets, and the ability to combine them. This is especially true in the area of natural language processing, which is where I think machine learning has the most potential.

I was really excited
2: The coolest thing right now in NLP is a whole bunch of them. They're trying to understand where and how the brain works, and how the brain processes and processes information, and they're trying to figure out how words get sent and received in our brains. So at the moment, we
3: The coolest thing right now in NLP is the idea of using data to predict behavior. We've been using this for years, and I think in the future it wi

## Text-to-SQL Task Setup

First, let's download the data of text-to-SQL pairs and the database against which we'll execute queries to retrieve answers.

The code below initializes the database and does some initial preprocessing data preprocessing + splitting for you.

In [15]:
%%capture
from urllib.request import urlretrieve

for filename in ("geography.json", "geography-db.sql"):
    urlretrieve(f"https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/{filename}", filename)

import re
import sqlite3
import json
from copy import deepcopy

DATABASE_NAME = 'geo.db'
SQL_FILENAME = 'geography-db.sql'
DATASET_FILENAME = 'geography.json'

with open(SQL_FILENAME, 'r') as file:
    sql_script = file.read()
    sql_script = re.sub(r"\s*ENGINE=[^ ]+","", sql_script)
    sql_script = re.sub(r"\s*DEFAULT CHARSET=[^ ;]+","", sql_script)
    sql_script = re.sub(r"\s*LOCK TABLES `[^`]+` WRITE;","", sql_script)  # remove LOCK TABLES
    sql_script = re.sub(r"\s*UNLOCK TABLES;","", sql_script)  # remove UNLOCK TABLES
    sql_script = sql_script.replace('`', '')  # remove backticks

# Connect to the SQLite database (this will create the file if it doesn't exist)
connection = sqlite3.connect(DATABASE_NAME)
print(sql_script)

connection.executescript(sql_script)
connection.commit()
connection.close()

connection = sqlite3.connect(DATABASE_NAME)
cursor = connection.cursor()
with open(DATASET_FILENAME, 'r') as file:
    dataset = json.load(file)

splits = {'train': [], 'dev': [], 'test': []}
for query_type in dataset:
    for example in query_type['sentences']:
        split = example['question-split']
        example['question'] = example['text']
        for key, value in example['variables'].items():
            example['question'] = example['question'].replace(key, value)
        example['sql'] = deepcopy(query_type['sql'])
        example['sql'] = example['sql'][0]
        for key, value in example['variables'].items():
            example['sql'] = example['sql'].replace(key, value)
        try:
            cursor.execute(example['sql'])
        except:
            continue
        example['db_answer'] = cursor.fetchall()
        del example['text']
        del example['variables']
        splits[split].append(example)

We also provide a function you can use to query the database:

In [16]:
def query_db(sql):
    connection = sqlite3.connect(DATABASE_NAME)
    cursor = connection.cursor()
    cursor.execute(sql)
    result = cursor.fetchall()
    connection.close()
    return result

This dataset is pretty small:

In [19]:
print('Train set size:', len(splits['train']))
print('Dev set size:', len(splits['dev']))
print('Test set size:', len(splits['test']))

Train set size: 547
Dev set size: 48
Test set size: 277


Let's inspect an example from the training dataset:

In [20]:
splits['train'][0]

{'question-split': 'train',
 'question': 'what is the biggest city in nebraska',
 'sql': 'SELECT CITYalias0.CITY_NAME FROM CITY AS CITYalias0 WHERE CITYalias0.POPULATION = ( SELECT MAX( CITYalias1.POPULATION ) FROM CITY AS CITYalias1 WHERE CITYalias1.STATE_NAME = "nebraska" ) AND CITYalias0.STATE_NAME = "nebraska" ;',
 'db_answer': [('omaha',)]}

Note that the `db_answer` is the result of executing the given SQL output against the database:

In [21]:
query_db(splits['train'][0]['sql'])

[('omaha',)]

Let's check how well our language model does on this text-to-SQL task out of the box. You can just use greedy decoding. 

In [43]:
prompt = "Write a SQL query based on the following question.\n\nQuestion: {input}\n\nSQL:"

In [44]:
# Your code here. Generate from the model using greedy decoding with the above prompt
question = splits['train'][0]['question']
promptEnc = tokenizer(prompt.format(input=question), return_tensors="pt").to(device)
output = gpt2_model.generate(**promptEnc, max_new_tokens=128, do_sample=False,pad_token_id=tokenizer.eos_token_id,)[0]
new_tokens = output[promptEnc.input_ids.shape[1]:]
predicted_sql =tokenizer.decode(new_tokens, skip_special_tokens=True)
predicted_sql = predicted_sql.strip().split("\n")[0]  # keep only the first line
print(predicted_sql)

SELECT city FROM city WHERE city.name = 'Nebraska' AND city.zipcode = '9'


You should get something that looks kind of like a SQL query, but it probably won't match the correct output, and in fact it most likely won't even execute without crashing when you try to query the database (you'll see a syntax error below).

In [45]:
try:
    query_db(predicted_sql)
    print('success!')
except:
    print('failed to execute!')

failed to execute!


Let's confirm quantitatively that the model doesn't work well out-of-the-box by running on the dev dataset (`splits['dev']`).

In [ ]:
@torch.no_grad()
def predict_greedy(model, data, max_new_tokens=128):
    """
    Return the model's greedy text-to-sql predictions on the given data split.
    The maximum number of new tokens generated (NOT including tokens in the prompt) should be equal to max_new_tokens.
    For speed, you should batch the generation. The tokenizer can handle multiple inputs simultaneously,
    but you'll need to tell it to pad using padding=True, and you may also need to set tokenizer.padding_side='left'.
    Hint: as a postprocessing step after you're done, you may need to cut off the output at the first appearance of '\n' if the output is continuing past the end of the SQL.
    """
    prompts = [d['question'] for d in data]
    predicted_sqls = []
    # Your code here
    batch_size=8
    for i in range(0,len(questions),batch_size):
        questionsBatch = questions[i:i+batch_size]
        questionsBatch = [prompt.format(input=question) for question in questionsBatch]
        inputsBatch = tokenizer(questionsBatch, padding=True, return_tensors="pt", padding_side = 'left').to(device)
        outputBatch = model.generate(**inputsBatch, max_new_tokens=max_new_tokens, do_sample=False,pad_token_id=tokenizer.eos_token_id,)
        # outputbatch = [outputBatch[i][inputsBatch[i].input_ids.shape[1]:] for i in range(len(outputBatch))]
        outputBatch = outputBatch[:, inputsBatch.input_ids.shape[1]:]
        outputDecode = tokenizer.batch_decode(outputBatch,skip_special_tokens=True)
        outputDecode = [output.strip().split("\n")[0] for output in outputDecode]
        predicted_sqls.extend(outputDecode)

    return predicted_sqls # list of strings containing SQL predictions for each question in the data

In [48]:
def check_execution_accuracy(predictions, data):
    assert len(predictions) == len(data)
    correct = 0
    for p, d in zip(predictions, data):
        try:
            if query_db(p) == d['db_answer']:
                correct += 1
        except: # failed to execute
            pass
    return correct / len(predictions)

In [54]:
predictions = predict_greedy(gpt2_model, splits['dev'])
print('example prediction:', predictions[0])
print('initial execution acc', check_execution_accuracy(predictions, splits['dev']))

example prediction: SELECT city FROM city WHERE city.name = 'Arizona' AND city.zipcode = 'AZ'
initial execution acc 0.0


You will probably observe an accuracy around 0-2%. (It may be hard to verify if your `predict_greedy` function is correct at this stage, because the expected accuracy is so low, but you will reuse it later with an improved model, at which point it will be more obvious if your implementation is correct.)

## Model Finetuning

Now let's prepare our dataset for finetuning (i.e., training our pretrained language model on this text-to-SQL training set). For each element in the dataset, it should have a text prompt and then the SQL output, similar to above. Your job is to fill in the labels field below. This field sets the labels to use for training during the language modeling task.

For the labels, we only want to train the model to output the text after the word "SQL:". This is because in the prompt, everything before the word "SQL:" will also be provided to the model as input. Hint: use -100 as the label for tokens you do not want to train on. Hint 2: When doing LM training, the labels are the same as the input tokens, except shifted to the left by one. You should check whether Huggingface is already doing the shifting, or whether you need to do the shifting yourself.

One thing to be careful of with all LMs is to make sure there are not extra spaces. So, the text should be formatted as like "SQL: {sql output}" not "SQL: {sql output} ". 

In [55]:
class Text2SQLDataset(Dataset):
    PROMPT = "Write a SQL query based on the following question.\n\nQuestion: {question}\n\nSQL: {sql}"

    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer
        tokenizer.padding_side = 'right'

        self.input_ids = []
        self.attn_masks = []
        self.labels = []

        training_texts = []
        for example in self.data:
            training_text = Text2SQLDataset.PROMPT.format(question=example['question'], sql=example['sql']) + "<|endoftext|>" # include the end token so model knows when to stop!
            training_texts.append(training_text)
        encodings_dict = self.tokenizer(training_texts, padding=True, truncation=True)
        for i, (example, training_text) in enumerate(zip(data, training_texts)):
            self.input_ids.append(torch.tensor(encodings_dict['input_ids'][i]))
            self.attn_masks.append(torch.tensor(encodings_dict['attention_mask'][i]))
            # Your code here
            label = self.input_ids[-1].clone()
            previousPart= Text2SQLDataset.PROMPT.format(question=example['question'], sql="").strip()
            previousPart = self.tokenizer(previousPart, truncation=True).input_ids
            
            label[:len(previousPart)]=-100
            label[self.attn_masks[-1]==0]=-100
            self.labels.append(label)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.attn_masks[idx], self.labels[idx]

In [60]:
train_dataset = Text2SQLDataset(splits['train'], tokenizer)
dev_dataset = Text2SQLDataset(splits['dev'], tokenizer)
test_dataset = Text2SQLDataset(splits['test'], tokenizer)

Use the Hugging Face Trainer to fine-tune GPT-2 Medium for 3 epochs. Set the per-device training batch size to 2, gradient accumulation to 8, warmup steps to 30, weight decay to 0.05, and eval batch size to 8. Pass both `**TRAINER_DEVICE_ARGS` and `**TRAINER_SAVE_ARGS` to `TrainingArguments`. If prompted for WandB, set `report_to="none"`.

The cell saves every 20 updates and resumes from the latest completed checkpoint when rerun.

In [61]:
from transformers import TrainerCallback

CHECKPOINT_DIR = OUTPUT_DIR / "hw5-checkpoints"
TRAINER_SAVE_ARGS = {
    "output_dir": str(CHECKPOINT_DIR),
    "save_strategy": "steps",
    "save_steps": 20,
    "save_total_limit": 2,
    "save_only_model": False,
}


class MarkCheckpointComplete(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        if args.should_save:
            checkpoint = Path(args.output_dir) / f"checkpoint-{state.global_step}"
            (checkpoint / ".complete").touch()


def latest_checkpoint():
    marker = max(CHECKPOINT_DIR.glob("checkpoint-*/.complete"),
                 key=lambda path: int(path.parent.name.split("-")[-1]), default=None)
    return str(marker.parent) if marker else None


# Your code here: set up TrainingArguments and create a Trainer named trainer.
from transformers import TrainingArguments, Trainer
def data_collator(batch):
    return {
        "input_ids":      torch.stack([ids for ids, mask, labels in batch]),
        "attention_mask": torch.stack([mask for ids, mask, labels in batch]),
        "labels":         torch.stack([labels for ids, mask, labels in batch]),
    }

args = TrainingArguments(
                        per_device_train_batch_size=2, 
                        num_train_epochs=3, 
                        gradient_accumulation_steps=8, 
                        warmup_steps=30, 
                        weight_decay=0.05, 
                        per_device_eval_batch_size=8, 
                        report_to="none", 
                        **TRAINER_DEVICE_ARGS, 
                        **TRAINER_SAVE_ARGS,)
trainer = Trainer(model=gpt2_model, args=args, data_collator=data_collator, train_dataset=train_dataset)


trainer.add_callback(MarkCheckpointComplete)
trainer.train(resume_from_checkpoint=latest_checkpoint())

Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.12it/s]


TrainOutput(global_step=105, training_loss=0.4056480044410342, metrics={'train_runtime': 69.0006, 'train_samples_per_second': 23.782, 'train_steps_per_second': 1.522, 'total_flos': 898920600772608.0, 'train_loss': 0.4056480044410342, 'epoch': 3.0})

Reload the final checkpoint using `latest_checkpoint()` and move it with `.to(device)`. You may need to delete the previously loaded model if you run out of device memory.

In [62]:
del gpt2_model
# Your code here
finetuned_model = AutoModelForCausalLM.from_pretrained(latest_checkpoint()).to(device)

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 714.94it/s]


Let's check our finetuned model's performance.

In [65]:
finetuned_predictions = predict_greedy(finetuned_model, splits['test'])
print('finetuned execution acc:', check_execution_accuracy(finetuned_predictions, splits['test']))

finetuned execution acc: 0.4981949458483754


You should achieve an accuracy of roughly 50% using the suggested training hyperparameters; we will check >40% in the autograder.

Save your predictions.

In [66]:
def save_predictions(predictions, filename):
    path = OUTPUT_DIR / filename
    temporary = path.with_suffix('.tmp')
    temporary.write_text('\n'.join(predictions), encoding='utf-8')
    temporary.replace(path)

save_predictions(finetuned_predictions, 'finetuned_predictions.txt')

Inspect some of your predictions compared to the correct outputs, and describe some common types of errors in your report. What fraction of errors are due to failing to execute (e.g., syntax error), and what fraction are due to executing but getting the wrong answer?

In [67]:
# Your code here
correct, exec_errors, wrong_answers = [], [], []

for pred, ex in zip(finetuned_predictions, splits['test']):
    try:
        result = query_db(pred)
    except Exception as e:                       # failed to execute (syntax error, bad column, ...)
        exec_errors.append((pred, ex, f"{type(e).__name__}: {e}"))
        continue
    if result == ex['db_answer']:
        correct.append((pred, ex))
    else:                                        # executed, but returned the wrong rows
        wrong_answers.append((pred, ex, result))

n_errors = len(exec_errors) + len(wrong_answers)
print(f"Total: {len(finetuned_predictions)} | correct: {len(correct)} | errors: {n_errors}")
print(f"Failed to execute: {len(exec_errors)} ({len(exec_errors) / n_errors:.1%} of errors)")
print(f"Executed but wrong: {len(wrong_answers)} ({len(wrong_answers) / n_errors:.1%} of errors)")

def show(title, items, k=5):
    print(f"\n{'=' * 20} {title} {'=' * 20}")
    for item in items[:k]:
        pred, ex, info = item
        print("Question :", ex['question'])
        print("Predicted:", pred)
        print("Gold     :", ex['sql'])
        print("Detail   :", info if isinstance(info, str) else f"got {info[:5]} expected {ex['db_answer'][:5]}")
        print("-" * 60)

show("FAILED TO EXECUTE", exec_errors)
show("EXECUTED BUT WRONG ANSWER", wrong_answers)

# Group execution errors by message to see which failure types are most common
from collections import Counter
print("\nMost common execution errors:")
for msg, count in Counter(e.split(":")[0] + ":" + e.split(":")[1][:40] for _, _, e in exec_errors).most_common(5):
    print(f"  {count:3d}  {msg}")

Total: 277 | correct: 138 | errors: 139
Failed to execute: 19 (13.7% of errors)
Executed but wrong: 120 (86.3% of errors)

==================== FAILED TO EXECUTE ====================
Question : what is the least populous state
Predicted: SELECT STATEalias0.LOWEST_ELEVATION FROM STATE AS STATEalias0 WHERE STATEalias0.STATE_NAME = "minnesota" ;
Gold     : SELECT STATEalias0.STATE_NAME FROM STATE AS STATEalias0 WHERE STATEalias0.POPULATION = ( SELECT MIN( STATEalias1.POPULATION ) FROM STATE AS STATEalias1 ) ;
Detail   : OperationalError: no such column: STATEalias0.LOWEST_ELEVATION
------------------------------------------------------------
Question : how long is the delaware river
Predicted: SELECT DERIVED_TABLEalias0.LENGTH FROM DERIVED_TABLE AS DERIVED_TABLEalias0 WHERE DERIVED_TABLEalias0.STATE_NAME = "delaware" ;
Gold     : SELECT DISTINCT RIVERalias0.LENGTH FROM RIVER AS RIVERalias0 WHERE RIVERalias0.RIVER_NAME = "delaware" ;
Detail   : OperationalError: no such table: DERIVED_TABL

We can also examine the exact match accuracy (i.e., requiring the predicted SQL string to exactly match the gold answer) rather than the execution accuracy (just checking whether the output of executing the SQL against the database is the same).

In [68]:
def check_exact_match_accuracy(predictions, data):
    assert len(predictions) == len(data)
    correct = 0
    for p, d in zip(predictions, data):
        if p == d['sql']:
            correct += 1
    return correct / len(predictions)

In [69]:
print('finetuned exact match acc:', check_exact_match_accuracy(finetuned_predictions, splits['test']))

finetuned exact match acc: 0.44765342960288806


The exact match accuracy will likely be close to the execution accuracy, but not exactly the same. What are some potential pros and cons of each metric? Discuss in your report.

Unload your finetuned model so that you don't run out of device memory later.

In [70]:
del finetuned_model

## Few-Shot Prompting

For the final part of this project, you will explore few-shot prompting, i.e., simply prompting the pretrained language model out-of-the-box using a small number of examples rather than finetuning.

First, let's try just selecting 4 examples completely at random from the training set. Rewrite your `predict_greedy` function to change the prompt:

In [72]:
import random


few_shot_prompt = "Question: {question0}\n\nSQL: {sql0}\n\n\n\n" + \
    "Question: {question1}\n\nSQL: {sql1}\n\n\n\n" + \
    "Question: {question2}\n\nSQL: {sql2}\n\n\n\n" + \
    "Question: {question3}\n\nSQL: {sql3}\n\n\n\n" + \
    "Question: {question}\n\nSQL:"


def select_random_examples(question, few_shot_data, num_examples=4):
    """
    Return a list containing 4 of the elements of few_shot_data, selected randomly
    """
    return random.sample(few_shot_data, num_examples)


@torch.no_grad()
def predict_greedy_fewshot(model, data, few_shot_data, max_new_tokens=128, example_selection_method=select_random_examples):
    """
    Return the model's greedy text-to-sql predictions on the given data split.
    The maximum number of new tokens generated (NOT including tokens in the prompt) should be equal to max_new_tokens.
    The four examples with their SQL outputs should go in {question1}, {sql1}, {question2}, {sql2}, etc. in the few_shot_prompt. 
    The final {question} is the question that we're currently evaluating on.
    """
    questions = [d['question'] for d in data]
    predicted_sqls = []
    prompts = []
    for question in questions:
        few_shot_examples = example_selection_method(question, few_shot_data, num_examples=4)
        prompts.append(few_shot_prompt.format(
            question0=few_shot_examples[0]['question'],
            sql0=few_shot_examples[0]['sql'],
            question1=few_shot_examples[1]['question'],
            sql1=few_shot_examples[1]['sql'],
            question2=few_shot_examples[2]['question'],
            sql2=few_shot_examples[2]['sql'],
            question3=few_shot_examples[3]['question'],
            sql3=few_shot_examples[3]['sql'],
            question=question
        ))
    # Your code here; should be fairly similar to your previous predict_greedy code.
    # Hint: if you batch, we recommend batch size 8-16.
    batch_size=8
    for i in range(0,len(prompts),batch_size):
        promptsBatch = prompts[i:i+batch_size]
        inputsBatch = tokenizer(promptsBatch, padding=True, return_tensors="pt", padding_side = 'left').to(device)
        outputBatch = model.generate(**inputsBatch, max_new_tokens=max_new_tokens, do_sample=False,pad_token_id=tokenizer.eos_token_id,)
        outputBatch = outputBatch[:, inputsBatch.input_ids.shape[1]:]
        outputDecode = tokenizer.batch_decode(outputBatch,skip_special_tokens=True)
        outputDecode = [output.strip().split("\n")[0] for output in outputDecode]
        predicted_sqls.extend(outputDecode)

    return predicted_sqls # list of strings containing SQL predictions for each question in the data

In [71]:
# Reload the gpt2 model if you need to, and move it with .to(device)
gpt2_model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2-medium").to(device)

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 344.40it/s]


In [73]:
# This call can take a few minutes even if you batch; you can debug on a subset of the dev set as needed.
predictions = predict_greedy_fewshot(gpt2_model, splits['test'], splits['train'])
print('4-shot prompting with random examples, execution acc:', check_execution_accuracy(predictions, splits['test']))

4-shot prompting with random examples, execution acc: 0.036101083032490974


You will probably observe between 0-5% accuracy. Random example selection doesn't work very well on this dataset. 

However, what if we select examples by picking the examples from the training set whose questions are most similar to our current question? To do this, load a pretrained sentence encoder, which takes a sentence as input and outputs a fixed-length vector encoding semantic information about that sentence. First compute the vectors associated with all the training set questions, and then select examples from the training set based on which question vectors have the largest dot products with the vector for your current question. 

In [75]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Your code here; load the sentence encoder (see https://www.sbert.net/ for documentation). A good choice of model is "all-MiniLM-L6-v2"
# Pass device=str(device) when constructing SentenceTransformer.

sentence_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=str(device)) 

def compute_question_encodings(data):
    """
    For each example in the data, add a field called 'question_encoding' to the example, which is the vector encoding of the question.
    """
    # Your code here
    vectors =sentence_encoder.encode([d['question'] for d in data], show_progress_bar=False) 
    # vectors = sentence_encoder.encode(data['question'], show_progress_bar=False)
    for example, vector in zip(data, vectors):
        example['question_encoding'] = np.array(vector)


def select_similar_examples(question, few_shot_data, num_examples=4):
    """
    Return a list containing 4 of the elements of few_shot_data, selected with questions most semantically similar to the given question. 
    The most similar question should be the LAST element of the list, second most similar should be the second to last element, etc.
    The reason is that in the few-shot prompt, **you want the best example to be the most recent one.**

    To rank by semantic similarity, first compute the vector for the current question, then compute its dot product with
    all training set vectors (hint: you may want to vectorize this computation using numpy). Then sort by dot product.

    You should take advantage of the 'question_encoding' field that you added to each example in compute_question_encodings.
    """
    # Your code here
    # hint: when you call .encode with your sentence encoder, use show_progress_bar=False to avoid tons of printouts
    curEncode = sentence_encoder.encode(question, show_progress_bar=False)
    dataEncode = np.stack([d['question_encoding'] for d in few_shot_data])
    scores = np.dot(dataEncode, curEncode)
    selected = np.argsort(scores)[-num_examples:]
    return [few_shot_data[i] for i in selected]



d:\Aa GXT Files\Code\EECS 283A\hw5\eecs183-283A-26fall-hw5\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\GXT\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6348.94it/s

In [76]:
# first precompute all the vectors for the training set
compute_question_encodings(splits['train'])

In [77]:
# this call will again take a few minutes, even if you batched; feel free to debug on smaller sets of dev
predictions = predict_greedy_fewshot(gpt2_model, splits['test'], splits['train'], example_selection_method=select_similar_examples)
print('4-shot prompting with similar examples, execution acc:', check_execution_accuracy(predictions, splits['test']))

4-shot prompting with similar examples, execution acc: 0.35018050541516244


You should now achieve about 34% accuracy. The autograder will check that you get >30%.

Save your predictions.

In [78]:
save_predictions(predictions, 'similar4shot_predictions.txt')

Once again, inspect some of your predictions (from prompting with similar examples) compared to the correct outputs, and describe some common types of errors in your report. Are there any differences compared to the finetuned model, or are the types of errors pretty similar? You can also check the exact match accuracy again.

In [79]:
from collections import Counter

def analyze_errors(preds, data, name, k=5):
    correct, exec_errors, wrong_answers = [], [], []
    for pred, ex in zip(preds, data):
        try:
            result = query_db(pred)
        except Exception as e:
            exec_errors.append((pred, ex, f"{type(e).__name__}: {e}"))
            continue
        (correct if result == ex['db_answer'] else wrong_answers).append((pred, ex, result))

    n_err = len(exec_errors) + len(wrong_answers)
    print(f"\n######## {name} ########")
    print(f"Execution acc: {len(correct) / len(data):.1%} | Exact match acc: {check_exact_match_accuracy(preds, data):.1%}")
    print(f"Errors: {n_err} | failed to execute: {len(exec_errors)} ({len(exec_errors) / n_err:.1%}) "
          f"| executed but wrong: {len(wrong_answers)} ({len(wrong_answers) / n_err:.1%})")

    print("Most common execution errors:")
    for msg, c in Counter(e.split(":")[0] + ":" + e.split(":")[1][:40] for *_, e in exec_errors).most_common(5):
        print(f"  {c:3d}  {msg}")

    for title, items in [("FAILED TO EXECUTE", exec_errors), ("EXECUTED BUT WRONG", wrong_answers)]:
        print(f"\n{'=' * 15} {title} {'=' * 15}")
        for pred, ex, info in items[:k]:
            print("Question :", ex['question'])
            print("Predicted:", pred)
            print("Gold     :", ex['sql'])
            print("Detail   :", info if isinstance(info, str) else f"got {info[:5]} expected {ex['db_answer'][:5]}")
            print("-" * 60)
    return {"correct": {ex['question'] for _, ex, _ in correct},
            "exec": exec_errors, "wrong": wrong_answers}

similar_predictions = predictions          # from cell 68 (similar 4-shot)
fewshot_res = analyze_errors(similar_predictions, splits['test'], "SIMILAR 4-SHOT (base GPT-2)")
finetune_res = analyze_errors(finetuned_predictions, splits['test'], "FINETUNED", k=0)   # summary only

# How much do the two methods agree on which questions they get right?
fs, ft = fewshot_res["correct"], finetune_res["correct"]
print(f"\nBoth correct: {len(fs & ft)} | only finetuned: {len(ft - fs)} | only few-shot: {len(fs - ft)} "
      f"| neither: {len(splits['test']) - len(fs | ft)}")


######## SIMILAR 4-SHOT (base GPT-2) ########
Execution acc: 35.0% | Exact match acc: 29.2%
Errors: 180 | failed to execute: 41 (22.8%) | executed but wrong: 139 (77.2%)
Most common execution errors:
   21  OperationalError: incomplete input
    9  OperationalError: no such column
    8  OperationalError: no such table
    2  OperationalError: near "RIVERAS"
    1  OperationalError: near "AS"

=============== FAILED TO EXECUTE ===============
Question : what is the area of florida
Predicted: SELECT AREA_SIZE FROM AREA AS AREA_SIZE WHERE AREA_SIZE > 150000 AND AREA_SIZE > 150000 AND AREA_SIZE > 150000 AND AREA_SIZE > 150000 AND AREA_SIZE > 150000 AND AREA_SIZE > 150000 AND AREA_SIZE > 150000 AND AREA_SIZE > 150000 AND AREA_SIZE > 150000 AND AREA_SIZE > 150000 AND AREA_SIZE > 150000 AND AREA_SIZE > 150000 AND AREA_SIZE > 150000 AND AREA_SIZE > 150000 AND AREA
Gold     : SELECT STATEalias0.AREA FROM STATE AS STATEalias0 WHERE STATEalias0.STATE_NAME = "florida" ;
Detail   : OperationalErr

# Bonus (optional)

Do some open-ended exploration to try to improve your performance on this dataset as much as possible (whether for finetuning or prompting). No hard requirement on how much to improve (or to improve at all), but please discuss the ideas you tried + how effective they were in your report. 150-300 words.

A non-exhaustive list of possible ideas:
* Use a different similarity metric for selecting examples in few-shot prompting
* Use more examples in few-shot prompting
* Load a different base model than GPT2-Medium, or look into calling the OpenAI API
* Tune the hyperparameters used for finetuning
* Try to combine few-shot prompting with finetuning

(Be careful with the GPU memory if you're using Kaggle/Colab, though- we're already nearly capping out the GPU memory in a few places with the current settings.)

In [ ]:
import gc
#free memory
for name in ["trainer", "gpt2_model", "finetuned_model", "model"]:
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
print(f"allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB | reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

allocated: 0.11 GB | reserved: 0.34 GB


In [81]:
import gc, time
from transformers import set_seed

BONUS_DIR = OUTPUT_DIR / "hw5-bonus"
bonus_results = []

def run_finetune(name, epochs=3, lr=5e-5, grad_accum=8, batch_size=2, warmup=30, weight_decay=0.05):
    set_seed(42)                                            # same initialization/shuffle for every run
    model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2-medium").to(device)
    args = TrainingArguments(
        output_dir=str(BONUS_DIR / name),                   # separate from hw5-checkpoints
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        warmup_steps=warmup,
        weight_decay=weight_decay,
        save_strategy="no",                                 # no checkpoints: runs are short, saves disk
        logging_steps=10,
        report_to="none",
        **TRAINER_DEVICE_ARGS,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_dataset, data_collator=data_collator)
    start = time.time()
    train_out = trainer.train()

    model.eval()
    dev_acc = check_execution_accuracy(predict_greedy(model, splits['dev']), splits['dev'])
    test_preds = predict_greedy(model, splits['test'])
    test_acc = check_execution_accuracy(test_preds, splits['test'])

    result = dict(name=name, epochs=epochs, lr=lr, grad_accum=grad_accum,
                  train_loss=round(train_out.training_loss, 4), dev_acc=dev_acc, test_acc=test_acc,
                  minutes=round((time.time() - start) / 60, 1))
    bonus_results.append(result)
    print(result)

    del trainer, model                                      # free GPU memory before the next run
    gc.collect(); torch.cuda.empty_cache()
    return test_preds

In [82]:
# Free the base model from the few-shot section if it's still loaded
if 'gpt2_model' in globals():
    del gpt2_model; gc.collect(); torch.cuda.empty_cache()

configs = [
    dict(name="baseline",       epochs=3,  lr=5e-5, grad_accum=8),   # the original settings, for reference
    dict(name="ep5",            epochs=5,  lr=5e-5, grad_accum=8),
    dict(name="ep8",            epochs=8,  lr=5e-5, grad_accum=8),
    dict(name="ep5_lr1e-4",     epochs=5,  lr=1e-4, grad_accum=8),
    dict(name="ep5_ga4",        epochs=5,  lr=5e-5, grad_accum=4),   # 2x more optimizer steps per epoch
]
bonus_preds = {c["name"]: run_finetune(**c) for c in configs}

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 435.38it/s]


Step,Training Loss
10,2.102108
20,0.886490
30,0.331148
40,0.202998
50,0.167070
60,0.107922
70,0.130804
80,0.103084
90,0.083795
100,0.103694


{'name': 'baseline', 'epochs': 3, 'lr': 5e-05, 'grad_accum': 8, 'train_loss': 0.4056, 'dev_acc': 0.6666666666666666, 'test_acc': 0.5054151624548736, 'minutes': 1.6}


Loading weights: 100%|██████████| 292/292 [00:00<00:00, 744.71it/s]


Step,Training Loss
10,2.102108
20,0.886490
30,0.331148
40,0.203006
50,0.167271
60,0.106796
70,0.132195
80,0.098433
90,0.081801
100,0.100255


{'name': 'ep5', 'epochs': 5, 'lr': 5e-05, 'grad_accum': 8, 'train_loss': 0.2652, 'dev_acc': 0.7083333333333334, 'test_acc': 0.6064981949458483, 'minutes': 2.3}


... (5 outputs left out)

{'name': 'ep5_lr1e-4', 'epochs': 5, 'lr': 0.0001, 'grad_accum': 8, 'train_loss': 0.2148, 'dev_acc': 0.75, 'test_acc': 0.6823104693140795, 'minutes': 2.2}


Loading weights: 100%|██████████| 292/292 [00:00<00:00, 792.88it/s]


Step,Training Loss
10,2.139421
20,0.958684
30,0.404829
40,0.248599
50,0.251060
60,0.174035
70,0.146053
80,0.127698
90,0.108993
100,0.128494


{'name': 'ep5_ga4', 'epochs': 5, 'lr': 5e-05, 'grad_accum': 4, 'train_loss': 0.176, 'dev_acc': 0.7291666666666666, 'test_acc': 0.6209386281588448, 'minutes': 2.3}


In [83]:
import pandas as pd
df = pd.DataFrame(bonus_results).sort_values("dev_acc", ascending=False)
display(df)

best = df.iloc[0]["name"]                                   # choose by DEV accuracy, not test
print("Best config by dev:", best)
save_predictions(bonus_preds[best], f"bonus_finetuned_{best}_predictions.txt")

,name,epochs,lr,grad_accum,train_loss,dev_acc,test_acc,minutes
2,ep8,8,0.00005,8,0.1757,0.750000,0.689531,3.2
3,ep5_lr1e-4,5,0.00010,8,0.2148,0.750000,0.682310,2.2
4,ep5_ga4,5,0.00005,4,0.1760,0.729167,0.620939,2.3
1,ep5,5,0.00005,8,0.2652,0.708333,0.606498,2.3
0,baseline,3,0.00005,8,0.4056,0.666667,0.505415,1.6


Best config by dev: ep8


## Submission

Write a separate `report.pdf` with your results and answers to the analysis questions above. **The report is required for credit.** A PDF export of the notebook does not count.

Submit these files:

* hw5.ipynb (this file)
* finetuned_predictions.txt
* similar4shot_predictions.txt
* report.pdf